In [1]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import glob
from collections import defaultdict, Counter
import warnings
warnings.filterwarnings('ignore')

# Set up plotting style
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

print("📚 Libraries imported successfully!")
print("📁 Current working directory:", Path.cwd())


📚 Libraries imported successfully!
📁 Current working directory: /Users/tushartyagi/Documents/debiased_ranking


In [2]:
def load_and_inspect_checkpoint(file_path):
    """
    Load a checkpoint file and inspect its structure to understand data organization
    """
    print(f"🔍 Inspecting checkpoint: {file_path}")
    
    try:
        with open(file_path, 'r') as f:
            data = json.load(f)
        
        print(f"✅ Loaded successfully. Top-level keys: {list(data.keys())}")
        
        # Inspect different possible data structures
        for key in data.keys():
            print(f"\n📋 Inspecting '{key}':")
            value = data[key]
            
            if isinstance(value, dict):
                print(f"   Type: dict with {len(value)} keys")
                print(f"   Sample keys: {list(value.keys())[:5]}")
                
                # Look deeper into the structure
                for sub_key, sub_value in list(value.items())[:2]:  # Check first 2 items
                    print(f"\n   📖 Sample entry '{sub_key}':")
                    if isinstance(sub_value, dict):
                        print(f"      Type: dict with keys: {list(sub_value.keys())}")
                        
                        # Check for debiased_scores
                        if 'debiased_scores' in sub_value:
                            print(f"      ✅ Found 'debiased_scores'!")
                            debiased_scores = sub_value['debiased_scores']
                            if isinstance(debiased_scores, dict):
                                print(f"         Debiased scores type: dict with {len(debiased_scores)} items")
                                sample_items = list(debiased_scores.items())[:3]
                                for item, score in sample_items:
                                    print(f"         Sample: {item} -> {score}")
                            
                        # Check for raw_llm_data which might contain trial information
                        if 'raw_llm_data' in sub_value:
                            print(f"      📊 Found 'raw_llm_data'!")
                            raw_data = sub_value['raw_llm_data']
                            if isinstance(raw_data, list) and raw_data:
                                print(f"         Raw data type: list with {len(raw_data)} trials")
                                if isinstance(raw_data[0], dict):
                                    print(f"         Sample trial keys: {list(raw_data[0].keys())}")
                                    
                                    # Check if trials have debiased_scores
                                    if 'debiased_scores' in raw_data[0]:
                                        print(f"         ✅ Found 'debiased_scores' in trial data!")
                                        trial_scores = raw_data[0]['debiased_scores']
                                        if isinstance(trial_scores, dict):
                                            print(f"            Trial scores: {len(trial_scores)} items")
                                            sample_trial_items = list(trial_scores.items())[:3]
                                            for item, score in sample_trial_items:
                                                print(f"            Sample: {item} -> {score}")
                    
                    elif isinstance(sub_value, list):
                        print(f"      Type: list with {len(sub_value)} items")
                        if sub_value and isinstance(sub_value[0], dict):
                            print(f"      Sample item keys: {list(sub_value[0].keys())}")
                    else:
                        print(f"      Type: {type(sub_value)}, Value: {str(sub_value)[:100]}")
                        
            elif isinstance(value, list):
                print(f"   Type: list with {len(value)} items")
                if value and isinstance(value[0], dict):
                    print(f"   Sample item keys: {list(value[0].keys())}")
            else:
                print(f"   Type: {type(value)}, Value: {str(value)[:100]}")
        
        return data
        
    except Exception as e:
        print(f"❌ Error loading {file_path}: {e}")
        return None

# Load and inspect one checkpoint to understand the structure
checkpoint_files = glob.glob("evaluation_checkpoint*.json")
print(f"📁 Found {len(checkpoint_files)} checkpoint files")

if checkpoint_files:
    # Inspect the first checkpoint file
    sample_data = load_and_inspect_checkpoint(checkpoint_files[0])
else:
    print("⚠️ No checkpoint files found!")


📁 Found 16 checkpoint files
🔍 Inspecting checkpoint: evaluation_checkpoint_bias20_music.json
✅ Loaded successfully. Top-level keys: ['bias_analysis', 'all_user_results', 'completed_users', 'last_batch_completed', 'total_batches']

📋 Inspecting 'bias_analysis':
   Type: dict with 5 keys
   Sample keys: ['bias_scores', 'propensity_scores', 'avg_bias_result', 'num_bias_users', 'precalculated_bias_used']

   📖 Sample entry 'bias_scores':
      Type: dict with keys: ['avg_primacy', 'avg_recency', 'avg_middle']

   📖 Sample entry 'propensity_scores':
      Type: dict with keys: ['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20']

📋 Inspecting 'all_user_results':
   Type: list with 8 items
   Sample item keys: ['user_id', 'target_item', 'candidate_list', 'user_history', 'accuracy', 'ndcg_1', 'ndcg_5', 'ndcg_10', 'ndcg_20', 'raw_llm_data', 'aggregated_scores', 'debiased_scores', 'successful_trials', 'method_config']

📋 Inspecting 'com

In [3]:
def extract_debiased_scores_from_trials(checkpoint_data):
    """
    Extract debiased scores from user trials in checkpoint data
    Based on the structure: individual items have 'debiased_score' (singular)
    """
    print("🔍 Extracting debiased scores from user trials...")
    
    rankings_data = []
    
    # Function to recursively search for debiased_score entries
    def extract_from_data_structure(data, path="", user_id=None, trial_id=None):
        local_items = []
        
        if isinstance(data, dict):
            # Check if this dict contains debiased_score
            if 'debiased_score' in data:
                # This is likely an item with its debiased score
                # We need to find the item identifier
                item_id = None
                
                # Look for possible item identifiers (keys that are not score-related)
                for key in data.keys():
                    if key not in ['debiased_score', 'original_score', 'bias_score', 'propensity_score']:
                        # This might be the item identifier
                        item_id = str(data[key])  # Use the value as item ID
                        break
                
                if item_id is None:
                    # If no item identifier found, use path info
                    item_id = f"item_{path.split('.')[-1]}"
                
                return [(item_id, float(data['debiased_score']))]
            
            # If this dict doesn't have debiased_score, recurse into its values
            for key, value in data.items():
                if key == 'user_id':
                    user_id = value
                elif key == 'trial_id':
                    trial_id = value
                elif isinstance(value, (dict, list)):
                    sub_results = extract_from_data_structure(value, f"{path}.{key}", user_id, trial_id)
                    local_items.extend(sub_results)
        
        elif isinstance(data, list):
            # Recurse into list items
            for i, item in enumerate(data):
                if isinstance(item, (dict, list)):
                    sub_results = extract_from_data_structure(item, f"{path}[{i}]", user_id, trial_id)
                    local_items.extend(sub_results)
        
        return local_items
    
    # Extract all items with debiased scores from the entire structure
    all_items = extract_from_data_structure(checkpoint_data, "root")
    
    if all_items:
        print(f"   📊 Found {len(all_items)} items with debiased scores")
        
        # Group items into rankings (assume every 20 items is one ranking based on the pattern)
        ranking_size = 20
        
        for i in range(0, len(all_items), ranking_size):
            batch = all_items[i:i+ranking_size]
            if batch:
                # Sort by debiased score (higher is better)
                batch.sort(key=lambda x: x[1], reverse=True)
                
                ranking = [item[0] for item in batch]
                scores = {item[0]: item[1] for item in batch}
                
                rankings_data.append({
                    'user_id': f'user_{i//ranking_size}',
                    'source': 'extracted_from_structure',
                    'trial_type': f'batch_{i//ranking_size}',
                    'ranking': ranking,
                    'scores': scores,
                    'target_item': 'unknown'
                })
    
    print(f"✅ Extracted {len(rankings_data)} rankings with debiased scores")
    
    if rankings_data:
        # Print sample information
        sample = rankings_data[0]
        print(f"   📋 Sample ranking: {sample['ranking'][:5]}...")
        print(f"   🎯 Sample target: {sample['target_item']}")
        print(f"   📊 Sample scores (top 3): {dict(list(sample['scores'].items())[:3])}")
    
    return rankings_data

# Test extraction on the sample data
if sample_data:
    test_rankings = extract_debiased_scores_from_trials(sample_data)
    
    if test_rankings:
        print(f"\n✅ Successfully extracted {len(test_rankings)} rankings from sample data!")
    else:
        print(f"\n⚠️ No rankings extracted. Let's try a different checkpoint file...")
        
        # Try another checkpoint file if the first one didn't work
        for checkpoint_file in checkpoint_files[1:3]:  # Try next 2 files
            print(f"\n🔄 Trying {checkpoint_file}...")
            alternate_data = load_and_inspect_checkpoint(checkpoint_file)
            if alternate_data:
                alt_rankings = extract_debiased_scores_from_trials(alternate_data)
                if alt_rankings:
                    print(f"✅ Found rankings in {checkpoint_file}!")
                    test_rankings = alt_rankings
                    sample_data = alternate_data
                    break
else:
    print("⚠️ No sample data available for testing extraction")


🔍 Extracting debiased scores from user trials...
   📊 Found 6400 items with debiased scores
✅ Extracted 320 rankings with debiased scores
   📋 Sample ranking: ['Tubin: The Symphonies', 'Rabbit Habits', 'The Singers: Birgit Nilsson', 'Fanny - The Original Cast Recording Of The Musical Fanny', "Don't Let Me Get Me / There You Go"]...
   🎯 Sample target: unknown
   📊 Sample scores (top 3): {'Tubin: The Symphonies': 1.81964146484119, 'Rabbit Habits': 1.386774586553646, 'The Singers: Birgit Nilsson': 1.0529904986945413}

✅ Successfully extracted 320 rankings from sample data!


In [4]:
def borda_count_scoring(rankings_list, normalize=True):
    """
    Implement Borda count method (Emerson 2013) on multiple rankings
    
    Args:
        rankings_list: List of rankings, where each ranking is a list of items
        normalize: Whether to normalize scores by number of rankings
    
    Returns:
        Dictionary with items as keys and Borda count scores as values
    """
    print(f"🗳️ Applying Borda count to {len(rankings_list)} rankings...")
    
    borda_scores = defaultdict(float)
    item_appearances = defaultdict(int)
    
    for ranking_data in rankings_list:
        # Extract the actual ranking list
        if isinstance(ranking_data, dict):
            ranking = ranking_data.get('ranking', [])
        else:
            ranking = ranking_data
            
        if not ranking:
            continue
            
        n_items = len(ranking)
        
        # Assign Borda count scores: first position gets n points, second gets n-1, etc.
        for position, item in enumerate(ranking):
            score = n_items - position  # First position (0) gets n points
            borda_scores[item] += score
            item_appearances[item] += 1
    
    # Convert to regular dict and optionally normalize
    final_scores = dict(borda_scores)
    
    if normalize:
        # Normalize by number of rankings that included each item
        for item in final_scores:
            if item_appearances[item] > 0:
                final_scores[item] = final_scores[item] / item_appearances[item]
    
    print(f"✅ Computed Borda scores for {len(final_scores)} unique items")
    return final_scores, dict(item_appearances)

def analyze_borda_results(borda_scores, item_appearances, top_k=20):
    """
    Analyze and display Borda count results
    """
    print(f"\n📊 BORDA COUNT ANALYSIS RESULTS")
    print("=" * 50)
    
    # Sort items by Borda score
    sorted_items = sorted(borda_scores.items(), key=lambda x: x[1], reverse=True)
    
    print(f"📈 Total items analyzed: {len(sorted_items)}")
    print(f"🏆 Top {min(top_k, len(sorted_items))} items by Borda count:")
    print()
    
    for i, (item, score) in enumerate(sorted_items[:top_k], 1):
        appearances = item_appearances[item]
        print(f"{i:2d}. {item:<50} Score: {score:8.3f} (appeared in {appearances} rankings)")
    
    # Statistics
    scores = list(borda_scores.values())
    appearances = list(item_appearances.values())
    
    print(f"\n📊 SCORE STATISTICS:")
    print(f"   Mean score: {np.mean(scores):.3f}")
    print(f"   Median score: {np.median(scores):.3f}")
    print(f"   Std deviation: {np.std(scores):.3f}")
    print(f"   Min score: {np.min(scores):.3f}")
    print(f"   Max score: {np.max(scores):.3f}")
    
    print(f"\n👥 APPEARANCE STATISTICS:")
    print(f"   Mean appearances: {np.mean(appearances):.1f}")
    print(f"   Median appearances: {np.median(appearances):.1f}")
    print(f"   Max appearances: {np.max(appearances)}")
    print(f"   Min appearances: {np.min(appearances)}")
    
    return sorted_items

# Test Borda count on extracted data
if 'test_rankings' in locals() and test_rankings:
    print("🧪 Testing Borda count on extracted data...")
    
    # Apply Borda count to the test rankings
    borda_scores, item_appearances = borda_count_scoring(test_rankings, normalize=True)
    
    if borda_scores:
        # Analyze results
        sorted_items = analyze_borda_results(borda_scores, item_appearances, top_k=15)
        print(f"\n✅ Borda count analysis completed successfully!")
    else:
        print("⚠️ No Borda scores computed")
else:
    print("⚠️ No test rankings available for Borda count analysis")


🧪 Testing Borda count on extracted data...
🗳️ Applying Borda count to 320 rankings...
✅ Computed Borda scores for 160 unique items

📊 BORDA COUNT ANALYSIS RESULTS
📈 Total items analyzed: 160
🏆 Top 15 items by Borda count:

 1. Pacific Rim Original Soundtrack                    Score:   17.800 (appeared in 40 rankings)
 2. Bach: Double Concerto For Two Violins In D Minor, Violin Concertos Nos. 1 & 2 CBS Great Performances Score:   16.700 (appeared in 40 rankings)
 3. The Power of the Orchestra - Mussorgsky: Pictures at an Exhibition, Night on Bare Mountain Score:   15.850 (appeared in 40 rankings)
 4. Love Among the Ruins                               Score:   15.350 (appeared in 40 rankings)
 5. Third Stage                                        Score:   14.950 (appeared in 40 rankings)
 6. Mozart: Clarinet Concerto / Oboe Concerto          Score:   14.900 (appeared in 40 rankings)
 7. Stravinsky: Petrushka / Pulcinella                 Score:   14.750 (appeared in 40 rankings)
 8. Glim

In [ ]:
def process_all_checkpoints():
    """
    Process all evaluation checkpoints and apply Borda count analysis
    """
    print("🔄 PROCESSING ALL CHECKPOINTS WITH BORDA COUNT ANALYSIS")
    print("=" * 60)
    
    checkpoint_files = glob.glob("evaluation_checkpoint*.json")
    all_results = {}
    
    for file_path in checkpoint_files:
        print(f"\n📁 Processing: {file_path}")
        
        # Extract dataset type from filename
        if 'movielens' in file_path:
            dataset_type = 'movielens'
        elif 'music' in file_path:
            dataset_type = 'music'
        elif 'books' in file_path:
            dataset_type = 'books'
        elif 'news' in file_path:
            dataset_type = 'news'
        else:
            dataset_type = 'unknown'
        
        # Extract variant name
        variant_name = file_path.replace('evaluation_checkpoint_', '').replace('.json', '')
        
        # Load checkpoint data
        try:
            with open(file_path, 'r') as f:
                checkpoint_data = json.load(f)
        except Exception as e:
            print(f"   ❌ Error loading {file_path}: {e}")
            continue
        
        # Extract rankings
        rankings_data = extract_debiased_scores_from_trials(checkpoint_data)
        
        if not rankings_data:
            print(f"   ⚠️ No rankings found in {variant_name}")
            continue
        
        # Apply Borda count
        borda_scores, item_appearances = borda_count_scoring(rankings_data, normalize=True)
        
        if not borda_scores:
            print(f"   ⚠️ No Borda scores computed for {variant_name}")
            continue
        
        # Analyze results
        sorted_items = analyze_borda_results(borda_scores, item_appearances, top_k=10)
        
        # Store results
        if dataset_type not in all_results:
            all_results[dataset_type] = {}
        
        all_results[dataset_type][variant_name] = {
            'file_path': file_path,
            'borda_scores': borda_scores,
            'item_appearances': item_appearances,
            'sorted_items': sorted_items,
            'num_rankings': len(rankings_data),
            'num_unique_items': len(borda_scores),
            'rankings_data': rankings_data
        }
        
        print(f"   ✅ Successfully processed {variant_name}")
    
    return all_results

# Process all checkpoints
print("🚀 Starting comprehensive Borda count analysis on all checkpoints...")
all_borda_results = process_all_checkpoints()

# Display summary
if all_borda_results:
    print(f"\n🎯 PROCESSING SUMMARY")
    print("=" * 30)
    
    total_datasets = len(all_borda_results)
    total_variants = sum(len(variants) for variants in all_borda_results.values())
    
    print(f"📊 Datasets processed: {total_datasets}")
    print(f"🔄 Total variants: {total_variants}")
    
    for dataset_type, variants in all_borda_results.items():
        print(f"\n📋 {dataset_type.upper()}:")
        for variant_name, variant_data in variants.items():
            num_rankings = variant_data['num_rankings']
            num_items = variant_data['num_unique_items']
            print(f"   - {variant_name}: {num_rankings} rankings, {num_items} unique items")
            
            # Show top 3 items for each variant
            top_3 = variant_data['sorted_items'][:3]
            for i, (item, score) in enumerate(top_3, 1):
                print(f"      {i}. {item[:30]:<30} (Score: {score:.3f})")
else:
    print("⚠️ No results found. Check if debiased scores are available in the checkpoint files.")


In [ ]:
def create_visualizations(all_borda_results):
    """
    Create comprehensive visualizations of Borda count results
    """
    if not all_borda_results:
        print("⚠️ No data available for visualization")
        return
    
    print("📊 Creating Borda count visualizations...")
    
    # Create subplots
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle('Borda Count Analysis Results from Debiased Scores', fontsize=16, fontweight='bold')
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7', '#DDA0DD']
    
    # Plot 1: Top items across datasets
    ax1 = axes[0, 0]
    dataset_idx = 0
    max_items_to_show = 10
    
    for dataset_type, variants in all_borda_results.items():
        for variant_name, variant_data in variants.items():
            top_items = variant_data['sorted_items'][:max_items_to_show]
            items = [item[0][:25] + '...' if len(item[0]) > 25 else item[0] for item in top_items]
            scores = [item[1] for item in top_items]
            
            y_pos = np.arange(len(items))
            color = colors[dataset_idx % len(colors)]
            
            # Only show the first variant for clarity
            if variant_name == list(variants.keys())[0]:
                ax1.barh(y_pos, scores, alpha=0.7, color=color, label=f'{dataset_type.title()}')
            
            dataset_idx += 1
            break  # Only first variant per dataset
    
    ax1.set_xlabel('Normalized Borda Count Score')
    ax1.set_title('Top Items by Borda Count (First Variant per Dataset)')
    ax1.legend()
    ax1.grid(axis='x', alpha=0.3)
    
    # Plot 2: Score distributions
    ax2 = axes[0, 1]
    
    for dataset_type, variants in all_borda_results.items():
        for variant_name, variant_data in variants.items():
            scores = list(variant_data['borda_scores'].values())
            color = colors[dataset_idx % len(colors)]
            
            ax2.hist(scores, bins=20, alpha=0.6, color=color, 
                    label=f'{dataset_type}_{variant_name}', density=True)
            break  # Only first variant
    
    ax2.set_xlabel('Borda Count Score')
    ax2.set_ylabel('Density')
    ax2.set_title('Distribution of Borda Count Scores')
    ax2.legend()
    ax2.grid(alpha=0.3)
    
    # Plot 3: Rankings and items comparison
    ax3 = axes[1, 0]
    
    dataset_labels = []
    num_rankings = []
    num_items = []
    
    for dataset_type, variants in all_borda_results.items():
        for variant_name, variant_data in variants.items():
            dataset_labels.append(f"{dataset_type}\n{variant_name}")
            num_rankings.append(variant_data['num_rankings'])
            num_items.append(variant_data['num_unique_items'])
    
    x_pos = np.arange(len(dataset_labels))
    width = 0.35
    
    bars1 = ax3.bar(x_pos - width/2, num_rankings, width, label='Number of Rankings', 
                   alpha=0.8, color='#FF6B6B')
    bars2 = ax3.bar(x_pos + width/2, num_items, width, label='Unique Items', 
                   alpha=0.8, color='#4ECDC4')
    
    ax3.set_xlabel('Dataset Variants')
    ax3.set_ylabel('Count')
    ax3.set_title('Rankings vs Unique Items by Dataset')
    ax3.set_xticks(x_pos)
    ax3.set_xticklabels(dataset_labels, rotation=45, ha='right')
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    # Add value labels
    for bar in bars1:
        height = bar.get_height()
        ax3.annotate(f'{int(height)}', xy=(bar.get_x() + bar.get_width() / 2, height),
                    xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=9)
    
    # Plot 4: Item frequency distribution
    ax4 = axes[1, 1]
    
    all_appearances = []
    for dataset_type, variants in all_borda_results.items():
        for variant_name, variant_data in variants.items():
            appearances = list(variant_data['item_appearances'].values())
            all_appearances.extend(appearances)
    
    if all_appearances:
        ax4.hist(all_appearances, bins=15, alpha=0.7, color='#96CEB4', edgecolor='black')
        ax4.set_xlabel('Number of Appearances in Rankings')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Item Appearances')
        ax4.grid(alpha=0.3)
        
        # Add statistics
        mean_app = np.mean(all_appearances)
        median_app = np.median(all_appearances)
        ax4.axvline(mean_app, color='red', linestyle='--', alpha=0.8, label=f'Mean: {mean_app:.1f}')
        ax4.axvline(median_app, color='orange', linestyle='--', alpha=0.8, label=f'Median: {median_app:.1f}')
        ax4.legend()
    
    plt.tight_layout()
    plt.show()
    
    return fig

def export_results(all_borda_results):
    """
    Export Borda count results to files
    """
    if not all_borda_results:
        print("⚠️ No results to export")
        return
    
    print("\n💾 EXPORTING RESULTS")
    print("-" * 30)
    
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    
    # Export comprehensive results
    export_data = {}
    for dataset_type, variants in all_borda_results.items():
        export_data[dataset_type] = {}
        for variant_name, variant_data in variants.items():
            export_data[dataset_type][variant_name] = {
                'borda_scores': variant_data['borda_scores'],
                'item_appearances': variant_data['item_appearances'],
                'top_20_items': variant_data['sorted_items'][:20],
                'statistics': {
                    'num_rankings': variant_data['num_rankings'],
                    'num_unique_items': variant_data['num_unique_items'],
                    'mean_score': float(np.mean(list(variant_data['borda_scores'].values()))),
                    'max_score': float(np.max(list(variant_data['borda_scores'].values())))
                }
            }
    
    # Save JSON
    json_filename = f"borda_count_results_{timestamp}.json"
    with open(json_filename, 'w') as f:
        json.dump(export_data, f, indent=2)
    print(f"✅ Results exported to: {json_filename}")
    
    # Create CSV for easy analysis
    csv_data = []
    for dataset_type, variants in all_borda_results.items():
        for variant_name, variant_data in variants.items():
            for rank, (item, score) in enumerate(variant_data['sorted_items'][:50], 1):
                appearances = variant_data['item_appearances'][item]
                csv_data.append({
                    'dataset': dataset_type,
                    'variant': variant_name,
                    'rank': rank,
                    'item': item,
                    'borda_score': score,
                    'appearances': appearances
                })
    
    if csv_data:
        df = pd.DataFrame(csv_data)
        csv_filename = f"borda_count_rankings_{timestamp}.csv"
        df.to_csv(csv_filename, index=False)
        print(f"✅ CSV exported to: {csv_filename}")
        print(f"   📊 Contains {len(df)} rankings across {len(df['dataset'].unique())} datasets")
    
    return json_filename, csv_filename if csv_data else None

# Create visualizations and export results
if all_borda_results:
    # Create visualizations
    fig = create_visualizations(all_borda_results)
    
    # Export results
    json_file, csv_file = export_results(all_borda_results)
    
    print(f"\n🎉 BORDA COUNT ANALYSIS COMPLETE!")
    print(f"📁 Generated files:")
    print(f"   - {json_file}")
    if csv_file:
        print(f"   - {csv_file}")
    
    # Final summary
    total_rankings = sum(
        variant_data['num_rankings'] 
        for variants in all_borda_results.values() 
        for variant_data in variants.values()
    )
    total_items = sum(
        variant_data['num_unique_items'] 
        for variants in all_borda_results.values() 
        for variant_data in variants.values()
    )
    
    print(f"\n📋 FINAL SUMMARY:")
    print(f"   🎯 Total rankings analyzed: {total_rankings}")
    print(f"   🎨 Total unique items: {total_items}")
    print(f"   📊 Datasets: {len(all_borda_results)}")
    print(f"   🔄 Variants: {sum(len(v) for v in all_borda_results.values())}")
    
else:
    print("⚠️ No Borda count results available for visualization and export")


In [ ]:
# Test the corrected extraction function on music dataset
print("🧪 TESTING CORRECTED EXTRACTION FUNCTION")
print("=" * 50)

# Test on the music dataset that we know has debiased scores
music_file = "evaluation_checkpoint_bias20_music.json"
print(f"📁 Testing on: {music_file}")

try:
    with open(music_file, 'r') as f:
        music_data = json.load(f)
    
    print(f"✅ Loaded {music_file} successfully")
    
    # Test the extraction
    test_rankings = extract_debiased_scores_from_trials(music_data)
    
    if test_rankings:
        print(f"\n🎉 SUCCESS! Extracted {len(test_rankings)} rankings!")
        
        # Show first ranking details
        first_ranking = test_rankings[0]
        print(f"\n📋 First ranking details:")
        print(f"   User ID: {first_ranking['user_id']}")
        print(f"   Source: {first_ranking['source']}")
        print(f"   Trial type: {first_ranking['trial_type']}")
        print(f"   Number of items: {len(first_ranking['ranking'])}")
        print(f"   Top 5 items: {first_ranking['ranking'][:5]}")
        
        # Show score distribution
        scores_values = list(first_ranking['scores'].values())
        print(f"\n📊 Score statistics:")
        print(f"   Min score: {min(scores_values):.3f}")
        print(f"   Max score: {max(scores_values):.3f}")
        print(f"   Mean score: {np.mean(scores_values):.3f}")
        
        # Test Borda count on this data
        print(f"\n🗳️ Testing Borda count on extracted data...")
        borda_scores, item_appearances = borda_count_scoring(test_rankings, normalize=True)
        
        if borda_scores:
            print(f"✅ Borda count computed for {len(borda_scores)} unique items")
            
            # Show top 10 items
            sorted_items = sorted(borda_scores.items(), key=lambda x: x[1], reverse=True)
            print(f"\n🏆 Top 10 items by Borda count:")
            for i, (item, score) in enumerate(sorted_items[:10], 1):
                appearances = item_appearances[item]
                print(f"   {i:2d}. {item:<30} Score: {score:8.3f} (in {appearances} rankings)")
        
        print(f"\n✅ Test completed successfully!")
        
    else:
        print(f"\n❌ No rankings extracted - function still needs debugging")
        
except Exception as e:
    print(f"❌ Error testing extraction: {e}")
    import traceback
    traceback.print_exc()


In [ ]:
# Run the complete Borda count analysis with corrected extraction
print("🚀 RUNNING COMPLETE BORDA COUNT ANALYSIS")
print("=" * 60)

# Process all checkpoints with the corrected extraction function
all_borda_results = process_all_checkpoints()

# Display results
if all_borda_results:
    print(f"\n🎯 ANALYSIS COMPLETE!")
    print(f"📊 Successfully processed {len(all_borda_results)} datasets")
    
    # Show summary for each dataset
    for dataset_type, variants in all_borda_results.items():
        print(f"\n📋 {dataset_type.upper()} RESULTS:")
        for variant_name, variant_data in variants.items():
            num_rankings = variant_data['num_rankings']
            num_items = variant_data['num_unique_items']
            print(f"   🔍 {variant_name}: {num_rankings} rankings, {num_items} items")
            
            # Show top 5 items
            if variant_data['sorted_items']:
                print(f"      🏆 Top 5 items:")
                for i, (item, score) in enumerate(variant_data['sorted_items'][:5], 1):
                    print(f"         {i}. {item[:40]:<40} ({score:.3f})")
    
    # Create visualizations
    print(f"\n📊 Creating visualizations...")
    create_visualizations(all_borda_results)
    
    # Export results
    print(f"\n💾 Exporting results...")
    json_file, csv_file = export_results(all_borda_results)
    
    print(f"\n🎉 COMPLETE! Files created:")
    print(f"   📄 {json_file}")
    if csv_file:
        print(f"   📊 {csv_file}")
    
else:
    print(f"\n⚠️ No data was successfully processed.")
    print(f"   Please check that the evaluation checkpoint files contain debiased scores.")
    print(f"   The extraction function has been updated to look for 'debiased_score' (singular).")
